# YEATS2–BTN 亲和力设计：Rosetta 界面 ΔΔG 流水线（终版）

流程：relax 产物 → 挑 WT 基准 → 逐突变建模（MutateResidue + 邻域重排）→ InterfaceAnalyzer → ΔΔG 汇总
特点：全函数化封装（可直接改造成 AI 智能体工具）、断点续跑（已完成的自动跳过）、输入输出确定性强

## 0. 配置区（唯一需要修改的地方）

In [ ]:
# ===== 唯一需要确认的配置 =====
ROSETTA_BIN  = "/home/hkl/rosetta.source.release-408/main/source/build/src/release/linux/6.18/64/x86/gcc/13/default/rosetta_scripts.default.linuxgccrelease"
WORK_DIR     = "/home/hkl/yeats2_work"
PARAMS_FILE  = "BTN.params"
MODEL_PREFIX = "wt_complex_clean"

CANDIDATES = {
    "WT":     ("284", "A", None),
    "EA284P": ("284", "A", "PRO"),
    "SA261E": ("261", "A", "GLU"),
    "HA259D": ("259", "A", "ASP"),
    "WA282P": ("282", "A", "PRO"),
}

import os
os.chdir(WORK_DIR)
print("工作目录:", os.getcwd())

## 1. 工具函数（智能体封装的接口层）

In [ ]:
import shutil, subprocess, glob
import pandas as pd

def read_rosetta_score(fn):
    """robust 解析 Rosetta 分数文件（兼容新旧两种格式）。
    输入: fn(str) 输出: DataFrame（数值列已转 float）"""
    lines = open(fn).read().splitlines()
    hdr_i = next(i for i, l in enumerate(lines) if "total_score" in l)  # 表头行
    header = lines[hdr_i].split()
    if header and header[0] == "SCORE:":
        header = header[1:]                                           # ← 关键修复
    rows = []
    for l in lines[hdr_i + 1:]:
        if not l.strip():
            continue
        p = l.split()
        if p and p[0] == "SCORE:":
            p = p[1:]
        if len(p) == len(header):
            rows.append(p)
    df = pd.DataFrame(rows, columns=header)
    for c in df.columns:
        if c != "description":
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def pick_best_model(prefix="wt_complex_clean", work_dir="."):
    """从 relax 分数文件中挑总能量最低的模型。输出: str 最优模型文件名"""
    cands = [f for f in glob.glob(os.path.join(work_dir, "*.sc"))
             if "score_" not in os.path.basename(f)]
    assert cands, "未找到分数文件 (.sc)"
    score_file = sorted(cands)[-1]
    sc = read_rosetta_score(score_file)
    sc = sc[sc["description"].str.startswith(prefix)]
    assert len(sc) > 0, f"{score_file} 中没有 {prefix} 模型"
    best = sc.sort_values("total_score").iloc[0]
    print(f"分数文件: {score_file} | 最优: {best['description']} ({best['total_score']:.1f})")
    return best["description"] + ".pdb"

def get_dg(fn):
    """读取界面结合能 dG_separated。输入: fn(str) 输出: float
    若列名不完全匹配会自动寻找最接近的列并报出实际列名"""
    df = read_rosetta_score(fn)
    col = next((c for c in df.columns if "dG" in c and "separated" in c.lower()), None)
    assert col, f"{fn} 中找不到 dG_separated 列，实际列名: {list(df.columns)}"
    return float(df[col].iloc[0])
    
def build_mutant_xml(pos, chain, new3):
    """生成 RosettaScripts XML：突变 → (约束下)FastRelax → (无约束权重)界面能。
    输入: pos(str) 残基号; chain(str) 链; new3(str|None) 新残基三字母，None=WT
    输出: str XML 文本"""
    TEMPLATE = """<ROSETTASCRIPTS>
  <SCOREFXNS>
    <ScoreFunction name="ref15" weights="ref2015"/>
    <ScoreFunction name="ref15_cst" weights="ref2015">
      <Reweight scoretype="coordinate_constraint" weight="1.0"/>
    </ScoreFunction>
  </SCOREFXNS>
  <MOVERS>
    <AddConstraints name="add_csts">
      <CoordinateConstraintGenerator name="cst" sd="0.5" native="yes"/>
    </AddConstraints>
__MUT__    <FastRelax name="relax" scorefxn="ref15_cst"/>
    <InterfaceAnalyzerMover name="iface" scorefxn="ref15" pack_input="0" pack_separated="0" interface="A_CD"/>
  </MOVERS>
  <PROTOCOLS>
    <Add mover="add_csts"/>
__A__    <Add mover="relax"/>
    <Add mover="iface"/>
  </PROTOCOLS>
</ROSETTASCRIPTS>
"""
    if new3:
        mut = '    <MutateResidue name="mut" target="{}{}" new_res="{}"/>\n'.format(pos, chain, new3)
        return TEMPLATE.replace("__MUT__", mut).replace("__A__", "    <Add mover=\"mut\"/>\n")
    return TEMPLATE.replace("__MUT__", "").replace("__A__", "")

def run_rosetta(tag, xml_text, input_pdb="wt_ref.pdb", params_file="BTN.params", resume=True):
    """执行 Rosetta 任务（幂等，可断点续跑）。输出: 分数文件路径"""
    xml_fn, sc_fn = f"ddg_{tag}.xml", f"score_{tag}.sc"
    if resume and os.path.exists(sc_fn):
        print(f"[{tag}] 已有结果，跳过")
        return sc_fn
    open(xml_fn, "w").write(xml_text)
    r = subprocess.run(
        [ROSETTA_BIN, "-s", input_pdb, "-in:file:native", input_pdb,
         "-parser:protocol", xml_fn,
         "-extra_res_fa", params_file, "-ex1", "-ex2",
         "-out:file:scorefile", sc_fn, "-overwrite"],
        capture_output=True, text=True)
    if "reported success" not in r.stdout:
        raise RuntimeError(f"[{tag}] 失败:\n{r.stdout[-1200:]}\n{r.stderr[-400:]}")
    print(f"[{tag}] 完成 ✓")
    return sc_fn

def check_btn(pdb, params="BTN.params"):
    """自检 BTN 重原子完整性"""
    pn = {l.split()[1] for l in open(params) if l.startswith("ATOM") and l.split()[1][0] in "CONS"}
    bn = {l[12:16].strip() for l in open(pdb) if l.startswith("HETATM") and l[17:20] == "BTN"}
    miss = pn - bn
    print(f"{pdb}: BTN {len(bn)}/{len(pn)} 缺失:{miss or '无 ✅'}")
    return not miss

## 2. 主流程：WT 基准 → 突变扫描 → ΔΔG 汇总

In [ ]:
# ===== 第 1 步：WT 基准 =====
best = pick_best_model(prefix=MODEL_PREFIX)
shutil.copy(best, "wt_ref.pdb")
assert check_btn("wt_ref.pdb"), "BTN 不完整，终止"

# ===== 第 2 步：逐个突变 =====
dg = {}
for tag, (pos, chain, new3) in CANDIDATES.items():
    print(f"--- [{tag}] ---")
    dg[tag] = get_dg(run_rosetta(tag, build_mutant_xml(pos, chain, new3)))

# ===== 第 3 步：ΔΔG 汇总 =====
wt = dg["WT"]
print(f"\nWT dG_separated = {wt:.2f}")
final = pd.DataFrame(
    [{"mutation": t, "dG_mut": round(dg[t], 2),
      "ddG_Rosetta": round(dg[t] - wt, 2)} for t in CANDIDATES if t != "WT"]
).sort_values("ddG_Rosetta").reset_index(drop=True)
print("\n==== Rosetta 界面 ΔΔG（负=亲和力提升，<-1 显著）====")
display(final)
final.to_csv("scan_output/ddG_rosetta_final.csv", index=False)
print("✅ 已存 scan_output/ddG_rosetta_final.csv")

## 3. 三证据合并（Scan 筛选 / FoldX 定量 / Rosetta 复算）

In [ ]:
if "final" not in dir():
    print("⚠️ 主流程未成功运行（见上一格报错），合并跳过")
else:
    try:
        fx = pd.read_csv("scan_output/ddG_withBTN.csv")[["mutation", "ddG_bind_withBTN"]]
        fx = fx.rename(columns={"ddG_bind_withBTN": "ddG_FoldX"})
        tbl = final.merge(fx, on="mutation", how="left")
        tbl["符号一致"] = tbl.apply(
            lambda r: "—" if pd.isna(r.ddG_FoldX) else ("✅" if r.ddG_Rosetta * r.ddG_FoldX > 0 else "❌"),
            axis=1)
    except FileNotFoundError:
        tbl = final.copy()
        print("ℹ️ 无 FoldX 结果，仅 Rosetta 数据")
    display(tbl)
    tbl.to_csv("scan_output/final_three_evidence.csv", index=False)

In [ ]:
import math, os

def all_res_atoms(pdb):
    out = {}
    for l in open(pdb):
        if l.startswith(("ATOM","HETATM")):
            out.setdefault((l[21], int(l[22:26])), {})[l[12:16].strip()] = (
                float(l[30:38]), float(l[38:46]), float(l[46:54]))
    return out

def mind(x, y):
    return min(math.dist(p, q) for p in x.values() for q in y.values())

relaxed = "wt_ref_0001.pdb"     # ← Rosetta 加 -out:pdb 后输出的突变模型
assert os.path.exists(relaxed), f"{relaxed} 不存在——需要先在 run_rosetta 加 -out:pdb 并重跑 WA282P"
res = all_res_atoms(relaxed)
pro = res[("A", 282)]
btn = {k: v for k, v in res[("D", 1)].items() if k[0] in "CONS"}
print("弛豫后 Pro282-BTN 重原子最近距离:", round(mind(pro, btn), 2), "Å")

## 4. 智能体工具封装指南

本 notebook 的四个函数即未来智能体的工具层，对应关系：

| 函数 | 工具描述 | 关键参数 |
|---|---|---|
| pick_best_model | 从 relax 结果中选择最优 WT 模型 | score_file, prefix |
| build_mutant_xml | 生成突变建模协议 | pos, chain, new3 |
| run_rosetta | 执行 Rosetta 计算（幂等，可断点续跑） | tag, xml_text, input_pdb, params_file |
| get_dg_separated | 读取界面结合能 | score_fn |

改造为 DeepSeek function calling 时，将每个函数的 docstring 转为
tools JSON 的 description，参数转为 properties（全部 string 类型即可），
执行循环：模型返回 tool_call → 本地执行对应函数 → 结果喂回模型。